# Laboratorio 04 — Buscar sin dar vuelta el almacén

**Duración: 60 minutos.**

En el laboratorio anterior cerramos con una pregunta. Valentina necesita el total de
**un mes**, y la tabla de la DGT tiene dos millones de documentos que crecen todos los
días.

> La tabla es un montón de archivos. Para encontrar un mes, **¿hay que abrirlos
> todos?**

Hoy la respondemos, y de paso vas a ver la diferencia más práctica que hay entre
Iceberg y el Hive que la DGT usa hoy.

---

### En el almacén

Una bodega sin orden obliga a abrir todas las cajas para encontrar los fideos. Una
bodega con **pasillos rotulados** no: vas al pasillo de fideos y listo. Eso son las
**particiones**.

Y hay dos cosas más, que son las que hacen la diferencia:

- **Tú no necesitas saber cómo ordenó la bodega el almacenero.** Pides fideos y él
  sabe a qué pasillo ir. Eso es el **particionamiento oculto**.
- **Si el almacén crece y hay que subdividir un pasillo, lo que ya estaba guardado se
  queda donde está.** No se vacía la bodega para reordenarla. Eso es la **evolución de
  particiones**.

Son tres cosas distintas y hoy las vas a ver por separado.

---

### Cómo se usa este notebook

Como siempre: `%%sql` en la primera línea, tú escribes debajo, una sentencia por
celda.

Los datos ya están en el ambiente, en `curso`: un año completo de documentos, 2024,
mes a mes. Tú lees de ahí y escribes en tu espacio, `mi_espacio`.

**Recuerda el nombre a tres partes** para las vistas del sistema:
`mi_espacio.documentos_sin_particion.files`, aunque ya hayas hecho `USE`.


## Paso 0 — Preparar la tabla

Primero, a tu espacio de trabajo.

In [ ]:
%%sql
-- Celda 0.1



Partimos de cero. Hoy vas a crear dos tablas, así que van dos borrados.

In [ ]:
%%sql
-- Celda 0.2



Y la segunda, que crearemos más adelante.

In [ ]:
%%sql
-- Celda 0.3



Ahora la primera tabla: un año de documentos, **sin particionar**. Es la tabla
que tendrías si nadie hubiera pensado en el problema.

In [ ]:
%%sql
-- Celda 0.4



Miremos con qué estamos trabajando. Es un año completo de documentos y las columnas
no son las mismas del laboratorio anterior.

In [ ]:
%%sql
-- Celda 0.5



¿Cuántos documentos quedaron?

In [ ]:
%%sql
-- Celda 0.6



## Paso 1 — La consulta lenta

Treinta mil documentos: los doce meses de 2024.

Ahora la consulta de Valentina. Quiere el total de **junio**, un mes de los doce.

In [ ]:
%%sql
-- Celda 1.1



Dos mil quinientos documentos. La respuesta es correcta y salió rápido, porque la
tabla es chica.

Pero la pregunta de hoy no es *cuánto demoró*: es **cuánto tuvo que leer**. Y eso sí
se puede mirar.

Empecemos por lo más simple: ¿en cuántos archivos está guardada esta tabla, y qué
rango de fechas abarca cada uno?

In [ ]:
%%sql
-- Celda 1.2



Ahí está el problema, y es más claro de lo que parece: **un solo archivo, con las
treinta mil filas, que abarca desde el 1 de enero hasta el 31 de diciembre.**

Iceberg guarda de cada archivo la fecha más chica y la más grande que contiene. Con
eso puede **descartar** archivos sin abrirlos: si buscas junio y un archivo va de
enero a marzo, ni lo toca.

Pero este archivo va de enero a diciembre. **Junio podría estar en cualquier parte de
él**, así que no se puede descartar: hay que leerlo entero.

Contemos exactamente lo que el motor no puede descartar. La condición de abajo es la
misma que usa él: *"el archivo termina después de que empieza junio, y empieza antes
de que junio termine"*, o sea, se solapa con junio.

In [ ]:
%%sql
-- Celda 1.3



## Paso 2 — Crear la tabla particionada

**Treinta mil filas para responder por dos mil quinientas.** Se lee el año entero
para contestar por un mes. Ese es el día a día de la DGT, en todas las consultas.

Vamos a construir la misma tabla, pero con pasillos. `PARTITIONED BY
(months(fecha_emision))` le dice a Iceberg: *guarda los documentos agrupados por el
mes de su fecha de emisión*.

Dos cosas de la sentencia:

- **`months(fecha_emision)` es una transformación.** No creas una columna nueva ni
  tienes que calcular el mes: le pasas la fecha real y Iceberg deriva el mes. Ese
  detalle es el que importa y lo vamos a mirar en el paso 4.
- **`ORDER BY fecha_emision` al final no sobra.** Para escribir en una tabla
  particionada, las filas tienen que llegar agrupadas por partición. Sin el `ORDER BY`
  la sentencia falla con un error sobre *"records are clustered"*. Es la forma normal
  de cargar una tabla particionada.

In [ ]:
%%sql
-- Celda 2.1



Comprobemos que quedaron los mismos documentos.

In [ ]:
%%sql
-- Celda 2.2



## Paso 3 — La misma consulta, ahora sí

Los mismos treinta mil. Las dos tablas tienen exactamente los mismos datos.

Ahora **la misma consulta de Valentina**, letra por letra igual a la del paso 1, sobre
la tabla nueva.

In [ ]:
%%sql
-- Celda 3.1



Mismo resultado: 2.500 documentos y el mismo total. Como debe ser — los datos son
los mismos.

Y ahora la medición, con **la misma consulta de archivos candidatos** del paso 1,
cambiando solo el nombre de la tabla.

In [ ]:
%%sql
-- Celda 3.2



## Paso 4 — El particionamiento oculto

Compara los dos números, que es a lo que vinimos:

| | archivos a leer | filas a leer |
|---|---|---|
| Sin particionar | 1 | **30.000** |
| Particionada por mes | 1 | **2.500** |

**Doce veces menos.** Y fíjate en la primera columna: la tabla particionada tiene
**más** archivos en total —doce, uno por mes— pero para esta consulta lee **uno**. Más
archivos, menos lectura. Con dos millones de documentos y años de historia, esa
proporción es la diferencia entre un reporte que sale y uno que no.

---

Ahora lo que de verdad separa a Iceberg del Hive que la DGT usa hoy. Vuelve a mirar
la consulta que escribiste:

```sql
WHERE fecha_emision BETWEEN DATE '2024-06-01' AND DATE '2024-06-30'
```

Filtraste por **`fecha_emision`**, la columna de verdad, la que le importa a
Valentina. **En ningún momento nombraste el mes.** Y aun así el motor supo ir a un
solo pasillo.

En Hive eso no pasaba. La tabla se particionaba por una columna aparte —`anio_mes`, o
como se llamara— y si tu consulta no la nombraba, se leía la tabla completa. Quien no
sabía cómo estaba particionada la tabla escribía consultas lentas sin enterarse.

Veamos cómo está particionada la tuya.

In [ ]:
%%sql
-- Celda 4.1



Busca la línea `Part 0`: dice **`months(fecha_emision)`**. Esa es toda la
especificación de partición.

Y fíjate en algo que no está: **no hay ninguna columna `mes` en la tabla.** El mes no
se guarda, se deriva. Por eso se llama *oculto*: existe para organizar, no para que tú
lo escribas.

Miremos los pasillos que se armaron.

In [ ]:
%%sql
-- Celda 4.2



## Paso 5 — Cambiar el criterio

Doce pasillos, uno por mes, 2.500 documentos cada uno.

---

Último acto, y es el que en Hive era **imposible** sin rehacer la tabla completa.

Supón que pasa lo que siempre pasa: el volumen crece. La DGT recibe ahora tantos
documentos que un mes ya es demasiado grueso, y conviene bajar a día. En Hive eso
significaba crear una tabla nueva, reescribir los años de historia y coordinar el
cambio con todos los que la consultan. Un proyecto.

Aquí es una sentencia. `REPLACE PARTITION FIELD` cambia el criterio **de aquí en
adelante**.

In [ ]:
%%sql
-- Celda 5.1



Veamos cómo quedó la especificación.

In [ ]:
%%sql
-- Celda 5.2



`Part 0` ahora dice **`days(fecha_emision)`**.

La pregunta importante: **¿y los treinta mil documentos que ya estaban?** ¿Se
reescribieron?

Carguemos el mes siguiente y miremos.

In [ ]:
%%sql
-- Celda 5.3



Ahora la respuesta. Cada archivo de la tabla recuerda **con qué criterio se
escribió**: eso es el `spec_id`. El 0 es el criterio viejo, por mes; el 1 es el nuevo,
por día.

In [ ]:
%%sql
-- Celda 5.4



Ahí está: **los doce archivos del criterio viejo siguen intactos**, con sus 30.000
documentos. No se tocó ni uno. Los 2.500 nuevos se guardaron con el criterio nuevo, en
31 archivos, uno por día.

Los dos criterios conviven en la misma tabla. Y ahora la prueba de que eso no rompió
nada: consultemos un mes del período **viejo**.

In [ ]:
%%sql
-- Celda 5.5



Y un mes del período **nuevo**.

In [ ]:
%%sql
-- Celda 5.6



---

## Listo

Las dos consultas respondieron, sin que tú tuvieras que saber que la tabla cambió de
criterio a mitad de camino. El motor sabe qué criterio usó cada archivo y usa el que
corresponde.

### Lo que te llevas

Tres cosas distintas, y conviene no mezclarlas:

| | Qué es | Cómo se ve |
|---|---|---|
| **Particionar** | Guardar los datos agrupados, para no leerlos todos | `PARTITIONED BY (months(fecha_emision))` |
| **Particionamiento oculto** | Filtras por la columna real; el motor sabe a qué pasillo ir | Tu consulta dice `fecha_emision`, nunca `mes` |
| **Evolución de particiones** | Cambiar el criterio sin reescribir lo que ya está | `ALTER TABLE … REPLACE PARTITION FIELD` |

Y el número que resume el laboratorio: **30.000 filas leídas contra 2.500** para
responder exactamente la misma pregunta.

### Dos cosas que conviene recordar del día que particiones de verdad

- **Al cargar una tabla particionada, las filas tienen que llegar agrupadas por
  partición.** De ahí el `ORDER BY fecha_emision` del paso 2.
- **Particionar crea más archivos.** Doce en vez de uno aquí; treinta y uno para un
  solo mes cuando bajamos a día. Eso tiene un costo propio, y es el tema del
  laboratorio 08.

### La pregunta del laboratorio siguiente

Tu tabla tiene diez columnas y todos los documentos las traen todas.

Pero el formato de los documentos tributarios **cambió**. Los que llegan desde 2026
traen dos campos que antes no existían: por qué canal se emitió el documento y en qué
sucursal. Los de 2024 no los tienen y nunca los van a tener.

Con lo que viste hoy ya sabes que la tabla no se reescribe para cambiar cómo se
guarda. **¿Pasará lo mismo con lo que se guarda?**

Eso es el **Laboratorio 05 — La columna que no existía**.
